# Big Data Analytics of NYC Yellow Taxi Trip Records Using Apache Spark

This interactive Jupyter notebook demonstrates the **PySpark pipeline** for ingesting, profiling, cleaning, transforming, and querying **4.11 GB** of NYC Yellow Taxi trip records.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum, round as spark_round

# 1. Initialize PySpark Session
spark = (SparkSession.builder
    .appName("NYC_Taxi_Notebook_Demo")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("Spark Session Created:", spark.version)

In [ ]:
# 2. Read Transformed Parquet Dataset
data_path = r"../processed_data/transformed_taxi_data"
df = spark.read.parquet(data_path)
df.cache()
print(f"Total Clean & Transformed Records: {df.count():,}")
df.printSchema()

In [ ]:
# 3. Execute PySpark SQL Query
df.createOrReplaceTempView("taxi_trips")
hourly_df = spark.sql("""
    SELECT 
        pickup_hour,
        COUNT(*) AS trip_count,
        ROUND(SUM(total_amount), 2) AS total_revenue_usd,
        ROUND(AVG(fare_amount), 2) AS avg_fare_usd
    FROM taxi_trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour ASC
""")
hourly_df.show(24)